### packages

In [ ]:
import numpy as np
import seaborn as sns
import pandas as pd
import itertools
import matplotlib.pyplot as plt
import pickle

from statsmodels.stats.multitest import multipletests
from scipy.stats import wilcoxon
from statannotations.Annotator import Annotator

import matplotlib

In [ ]:
# statistical annotation
def convert_pvalue_to_asterisks(pvalue):
    if pvalue <= 0.0001:
        return "****"
    elif pvalue <= 0.001:
        return "***"
    elif pvalue <= 0.01:
        return "**"
    elif pvalue <= 0.05:
        return "*"
    return "ns"

In [ ]:
def cal_mean_sem(values):
    values = [x for x in values if ~np.isnan(x)]
    # Calculate the mean
    mean_value = np.mean(values)
    # Calculate the SEM
    sem_value = sem(values)

    return mean_value, sem_value

In [ ]:
def truncate_colormap(cmap, minval=0.0, maxval=1.0, n=256):
    new_cmap = LinearSegmentedColormap.from_list(
        f'trunc({cmap.name},{minval:.2f},{maxval:.2f})',
        cmap(np.linspace(minval, maxval, n))
    )
    return new_cmap

### load config

In [ ]:
results_dir = '../../data/simulation_results/results_itrs_challenge'

In [ ]:
weight_list = np.arange(0, 1.1, 0.1, dtype=float)
agent_list = ['beh','weightedAverage','synergy'] #

In [ ]:
palette = sns.color_palette("tab10")
custom_linestyles = [(None,None),(5,5),(1,1)]
SPE_COLOR_DARK = '#c25269'
RPE_COLOR_DARK = '#1f7f8f'
SPE_COLOR = '#f7a6b6' 
RPE_COLOR = '#37b7c7' 

# Challenge 1

In [ ]:
# load results
with open(results_dir+'1.pkl', 'rb') as f:  # Python 3: open(..., 'wb')
    itr_df_1 = pickle.load(f)

with open(results_dir+'1_taus.pkl', 'rb') as f1:
    goal_trajectory, goal_trajectory1, valid_taus_agents, agent_goal_dists = pickle.load(f1)

with open(results_dir+'1_pes.pkl', 'rb') as f:  # Python 3: open(..., 'wb')
    itr_df_1_pes = pickle.load(f)

In [ ]:
# perfect decoding in deterministic env
prob_ = 1
noise_ = 0
this_df = itr_df_1[(itr_df_1['noise']==noise_ )&(itr_df_1['transprob']==prob_)]
cm = 1/2.54
fig, ax = plt.subplots(nrows=1, ncols=4, figsize=(4.5*4*cm,5*cm),\
                       sharex=True, sharey=True)
color = [palette[0], palette[1], palette[2], palette[4]]
weight_list = np.arange(0, 1.1, 0.1, dtype=float)

witrs = []
wplist = []
agentplist = []


for s in range(4):
    df = this_df[this_df['scene']==s+1]

    sns.lineplot(data=df,x='weight',y='itrs_x', hue='agent',ax=ax[s], errorbar='se',\
                 palette=[color[s], color[s], color[s]], style='agent',\
                    dashes={agent_list[0]: custom_linestyles[0], 
                 agent_list[1]: custom_linestyles[1], 
                 agent_list[2]: custom_linestyles[2]}) # 
    
    agent_w_p = [ ] 
    agent_w_itrs = []
    for a in agent_list[1:]:
        w_p = []
        w_itr = []
        df_1 = df[df['agent']==a]
        for w in weight_list[1:-1]:
            this_comp_0 = df_1[(df_1['weight']==w)]['itrs_x']
            this_comp_1 = df_1[(df_1['weight']==0)]['itrs_x']
            this_comp_2 = df_1[(df_1['weight']==1)]['itrs_x']
            w_itr.append([this_comp_0, this_comp_1, this_comp_2])

            _, p1 = wilcoxon(x=this_comp_0, y=this_comp_1, alternative='greater')
            _, p2 = wilcoxon(x=this_comp_0, y=this_comp_2, alternative='greater')
            
            w_p.append(p1)
            w_p.append(p2)
        _, w_p, _, _ = multipletests(w_p, method='fdr_bh')
        w_p = w_p.reshape((len(weight_list[1:-1]),2))
        
        agent_w_itrs.append(w_itr)
        
        # print(w_p)
        for w in range(len(w_p)):
            if all(w_p[w]<0.05):
                ax[s].scatter(weight_list[w+1], np.mean(df_1[(df_1['weight']==weight_list[w+1])]['itrs_x']),\
                            marker='o', edgecolor=color[s], facecolors='white', s=15).set_zorder(3)
        agent_w_p.append(w_p)
    
    wplist.append(agent_w_p)


    if s == 0:
        ax[s].set_ylabel('Adaptation speed', fontsize=10)
        # ax[s].set_ylabel(r'Learning speed [${log_{10}(1/itr)}$]', fontsize=8)
        # '\\textit{Velocity (\N{DEGREE SIGN}/sec)}'
    else:
        ax[s].set_xlabel('')

    # Show tick bars at all values (e.g. 0.0 to 1.0 in your weight_list)
    ax[s].set_xticks(weight_list)  # tick bars

    # Show labels only at desired ticks
    label_locs = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    ax[s].set_xticklabels([str(round(w, 1)) if round(w, 1) in label_locs else '' for w in weight_list],\
                           fontsize=10)

    
    # ax[s].set_yticks(ax[s].get_yticks(),[round(y,1) for y in ax[s].get_yticks()], fontsize=10)
    ax[s].set_yticks(label_locs,label_locs, fontsize=10)
    # ax[s].set_ylim([-0.05,1])
    ax[s].legend().remove()

# handles, labels = ax[s].get_legend_handles_labels()
# plt.legend(handles,['Q learning', 'Weighted average', 'Synergy'],ncols=1, title='')
plt.tight_layout(w_pad=.1)
plt.gcf().set_dpi(1000)
plt.show()
fig.savefig('../../results/fig6a.svg', transparent=True, format='svg')

In [ ]:
# Use only the darker part of the Greys colormap
from matplotlib import cm
from matplotlib.colors import LinearSegmentedColormap
cmap = truncate_colormap(cm.YlGn, 0.3, 1) # 3: truncate_colormap(cm.YlGn, 0.3, 1), 4: truncate_colormap(cm.BuPu, 0.3, 1)

from matplotlib import colors
# Use gamma < 1 for more dense color change at the beginning (e.g., gamma=0.5)
max_trials = np.max([len(valid_taus_agents[0]),len(valid_taus_agents[1])])
norm = colors.PowerNorm(gamma=0.5, vmin=0, vmax=max_trials - 1)

from matplotlib.cm import ScalarMappable


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12*1/2.54,6*1/2.54))

# plotting
for n in range(len(agent_list[1:])):
    ax = axes[n]

    valid_trajectories_ = valid_taus_agents[n]

    # Set up plot
    ax.set_xlim(0, 21)
    ax.set_ylim(0, 21)
    ax.set_aspect('equal')  # use set_aspect, not gca().set_aspect
    ax.grid(True, which='minor', linestyle='-', linewidth=0.2, color='gray', alpha=0.2)

    # Set major ticks at cell centers with labels 1 to 21
    ax.set_xticks(np.arange(0.5, 21.5, 1))
    ax.set_yticks(np.arange(0.5, 21.5, 1))
    # ax.set_xticklabels(range(1, 22))
    # ax.set_yticklabels(range(1, 22))
    # Remove all tick marks (major and minor)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    ax.tick_params(axis='both', which='both', length=0)

    # Set minor ticks at grid edges (excluding outermost)
    ax.set_xticks(np.arange(1, 21), minor=True)
    ax.set_yticks(np.arange(1, 21), minor=True)

    # Enable grid only on minor ticks (between cells)
    ax.grid(True, which='minor', linestyle='-', linewidth=0.5)
    ax.grid(False, which='major')  # Turn off grid on major ticks

    # === Draw only final goals and link them with arrows ===
    goal_points = []

    for i, trajectory in enumerate(valid_trajectories_):
        x_coords, y_coords = trajectory[0], trajectory[1]
        goal_x = x_coords[-1] - 0.5
        goal_y = y_coords[-1] - 0.5
        goal_points.append((goal_x, goal_y))

        # Color for this point
        b_color = cmap(norm(i))
        ax.scatter(goal_x, goal_y, marker='*', color=b_color, s=15)

    # print(goal_points)
    # Draw arrows between consecutive goals
    for i in range(len(goal_points) - 1):
        start = goal_points[i]
        end = goal_points[i + 1]
        ax.annotate(
            '', xy=end, xytext=start,
             arrowprops=dict(
                arrowstyle="->,head_length=0.3,head_width=0.1",  # smaller arrowhead
                color=cmap(norm(i + 1)),
                lw=0.8,
                alpha=0.8
            )
        )


    # === Goal trajectory with offset and transparency ===

    # Slight offset
    offset = 0
    gx, gy = goal_trajectory[0] + offset, goal_trajectory[1] + offset
    gx1, gy1 = goal_trajectory1[0] + offset, goal_trajectory1[1] + offset

    # Start and end markers
    ax.scatter(gx[0] - 0.5, gy[0] - 0.5, color='grey', s=10, alpha=0.7, label='Goal Start')
    ax.scatter(gx[-1] - 0.5, gy[-1] - 0.5, color='grey', marker='*', s=20, label='Previous goal')
    ax.scatter(gx1[-1]- 0.5, gy1[-1] - 0.5, color=cmap(norm(max_trials)), marker='*', s=20, label='Current goal')

    # Draw blue semi-transparent trajectory
    for i in range(len(gx) - 1):
        dx = gx[i+1] - gx[i]
        dy = gy[i+1] - gy[i]
        ax.plot([gx[i] - 0.5, gx[i] - 0.5+dx], [gy[i] - 0.5, gy[i] - 0.5+dy],
                color='grey',linewidth=1, alpha=0.7)
    # Draw red semi-transparent trajectory
    for i in range(len(gx1) - 1):
        dx = gx1[i+1] - gx1[i]
        dy = gy1[i+1] - gy1[i]
        ax.plot([gx1[i] - 0.5, gx1[i] - 0.5+dx], [gy1[i] - 0.5,  gy1[i] - 0.5+dy],
                color=cmap(norm(max_trials)), linewidth=1.5, alpha=0.7)
    
    ax.set_title(f'{agent_list[1:][n]}', fontsize=10)
    # Add number of valid trajectories as text box (like a second legend)
    ax.text(
        0.3, 0.15,  # position: upper right, slightly below the legend
        f'n = {len(valid_trajectories_)}',
        transform=ax.transAxes,
        fontsize=10,
        verticalalignment='center',
        horizontalalignment='center',
        bbox=dict(boxstyle='round', facecolor='white', edgecolor='gray', alpha=0.8)
    )


        
# plt.title("Multiple Trajectories in 21x21 GridWorld (Later = Darker)")
# plt.legend(["Start", "End"], loc='upper right')
# Add colorbar to the right of the figure
sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  # dummy for colorbar

# Add a colorbar axis on the right
cbar = fig.colorbar(sm, ax=axes, orientation='vertical', fraction=0.02, pad=0.04)
cbar.set_label('Trajectory order', fontsize=10)
cbar.ax.tick_params(labelsize=10)

# plt.tight_layout(w_pad=.1)
plt.gcf().set_dpi(1000)
fig.savefig('../../results/fig6b.svg', transparent=True, format='svg')
plt.show()

In [ ]:
# only goal distances
fig, ax1 = plt.subplots(1, 1, figsize=(5/2.54, 4.8/2.54))
m = 2
custom_linestyles = [(None,None),(5,5),(1,1)]


for a in range(len(agent_list[1:])):
    trials_goal, mean_goal, sem_goal = agent_goal_dists[a]
    line = ax1.plot(trials_goal, mean_goal, label=f'{agent_list[1:][a]}', color=palette[m])
    if custom_linestyles[a + 1] != (None, None):
        line[0].set_dashes(custom_linestyles[a + 1])
    ax1.fill_between(trials_goal, mean_goal - sem_goal, mean_goal + sem_goal, alpha=0.3, color=palette[m])
    
    agent_goal_dists.append([trials_goal, mean_goal, sem_goal])

# ax1.set_title('Weighted Levenshtein distance to goal trajectory', fontsize=18, fontweight='bold')
ax1.set_xlabel('Trial', fontsize=10)
ax1.set_yticks([0, 5,10,15,20], [0,5,10,15,20], fontsize=10)
ax1.set_xticks([0, 10, 20, 30], [0, 10, 20, 30], fontsize=10)
ax1.legend(loc='upper right').remove()


# plt.tight_layout(w_pad=.1)
plt.gcf().set_dpi(1000)
fig.savefig('../../results/fig6c.svg', transparent=True, format='svg')
plt.show()

In [ ]:
# perfect decoding_agent for 4 scenarios
cm = 1/2.54
noise_ = 0
w = 0.1
pes = ['rpe', 'spe']
fig, ax = plt.subplots(nrows=len(pes), ncols=4, figsize=(4.5*4*cm,4*len(pes)*cm),\
                       sharex=True, sharey=True)

color_pe = [RPE_COLOR, SPE_COLOR, 'black']
custom_linestyles = [(None,None),(5,5),(1,1)]

wplist = []
agentplist = []
witrs = []

for pe in range(len(pes)):
    for s in range(4):
        df = itr_df_1_pes[(itr_df_1_pes['noise']==noise_)&(itr_df_1_pes['scene']==s+1)\
                &(itr_df_1_pes['weight']==w)]

        sns.lineplot(data=df,x='trial',y=pes[pe], hue='agent',ax=ax[pe,s], errorbar='se',\
                        palette=[color_pe[pe], color_pe[pe], color_pe[pe]],\
                        style='agent',\
                        dashes={agent_list[0]: custom_linestyles[0], 
                    agent_list[1]: custom_linestyles[1], 
                    agent_list[2]: custom_linestyles[2]})
    

        if s == 0:
            ax[pe,s].set_ylabel('PEs', fontsize=10)
        else:
            ax[pe,s].set_xlabel('')

        ax[pe,s].legend().remove()
        

plt.tight_layout(w_pad=.1)
plt.gcf().set_dpi(1000)
plt.show()
fig.savefig('../../results/figS9a.svg', transparent=True, format='svg')

In [ ]:
# imperfect decoding_agent with certain acc 0.5
cm = 1/2.54
noise_ = 'CMsel'
prob_ = 1
weight_ = weight_list[1]

this_df = itr_df_1[(itr_df_1['noise']==noise_)\
                    &(itr_df_1['agent']!=agent_list[1])&(itr_df_1['transprob']==prob_)&\
                        (itr_df_1['weight']==weight_)]

fig, ax = plt.subplots(nrows=1, ncols=4, figsize=(4.5*4*cm,4.3*cm),\
                       sharex=True) 

custom_linestyles = [(None,None),(5,5),(1,1)]


wplist = []
agentplist = []
witrs = []

for s in range(4):
    df = this_df[this_df['scene']==s+1]

    sns.pointplot(data=df,x='agent',y='itrs_x', ax=ax[s],\
                 color=color[s],errorbar='se') #color[s], color[s], color[s] 'black','black','black'
    
    this_comp_1 = df[(df['agent']==agent_list[0])]['itrs_x']
    this_comp_2 = df[(df['agent']==agent_list[-1])]['itrs_x']
    _, p = wilcoxon(x=this_comp_1, y=this_comp_2, alternative='two-sided')

    annots = [convert_pvalue_to_asterisks(p)]
    pairs = [(agent_list[0], agent_list[-1])]
    
    annot = Annotator(ax=ax[s], pairs=pairs, data=df, x='agent',y='itrs_x')
    annot.configure(loc='inside', fontsize=12,text_offset=0, line_width=.8, verbose=0)
    annot.set_custom_annotations(annots)
    annot.annotate()


    if s == 0:
        ax[s].set_ylabel('Adaptation speed', fontsize=10)
        # ax[s].set_ylabel(r'Learning speed [${log_{10}(1/itr)}$]', fontsize=8)
        # '\\textit{Velocity (\N{DEGREE SIGN}/sec)}'
    else:
        ax[s].set_ylabel('')

    ax[s].set_xlabel('')
    ax[s].set_xlim([-0.4, 1.4])

    ax[s].set_xticks([0,1], ['Beh.', 'NVA-syn.'],\
                           fontsize=10)
    if s < 2:
        ax[s].set_yticks([0,0.05,0.10],[0,0.05,0.10], fontsize=10)
        ax[s].set_ylim([0, 0.12])
    else:
        ax[s].set_yticks([0,0.1,0.2,0.3],[0,0.1,0.2,0.3], fontsize=10) 
        ax[s].set_ylim([0, 0.25])

plt.tight_layout(w_pad=.1)
plt.gcf().set_dpi(1000)
plt.show()
fig.savefig('../../results/fig7.svg', transparent=True, format='svg')

In [ ]:
# perfect decoding_agent for 4 scenarios
cm = 1/2.54
noise_ = 'CMsel'
w = 0.1
pes = ['rpe', 'spe']
fig, ax = plt.subplots(nrows=len(pes), ncols=4, figsize=(4.5*4*cm,4*len(pes)*cm),\
                       sharex=True, sharey=True)

color = [RPE_COLOR, SPE_COLOR, 'black']
custom_linestyles = [(None,None),(5,5),(1,1)]

wplist = []
agentplist = []
witrs = []

for pe in range(len(pes)):
    for s in range(4):
        df = itr_df_1_pes[(itr_df_1_pes['noise']==noise_)&(itr_df_1_pes['scene']==s+1)\
                &(itr_df_1_pes['weight']==w)]

        sns.lineplot(data=df,x='trial',y=pes[pe], hue='agent',ax=ax[pe,s], errorbar='se',\
                        palette=[color[pe], color[pe]],\
                        style='agent',\
                        dashes={agent_list[0]: custom_linestyles[0], 
                    agent_list[1]: custom_linestyles[1], 
                    agent_list[2]: custom_linestyles[2]})
    

        if s == 0:
            ax[pe,s].set_ylabel('PEs', fontsize=10)
        else:
            ax[pe,s].set_xlabel('')

        ax[pe,s].legend().remove()
        

plt.tight_layout(w_pad=.1)
plt.gcf().set_dpi(1000)
plt.show()
fig.savefig('../../results/figS9b.svg', transparent=True, format='svg')

# Challenge 2

In [ ]:
# load results
with open(results_dir+'2.pkl', 'rb') as f:  # Python 3: open(..., 'wb')
    itr_df_2 = pickle.load(f)

In [ ]:
# perfect decoding in deterministic env
prob_ = 1
noise_ = 0
this_df = itr_df_2[(itr_df_2['noise']==noise_ )&(itr_df_2['transprob']==prob_)]
cm = 1/2.54
fig, ax = plt.subplots(nrows=1, ncols=4, figsize=(4.5*4*cm,5*cm),\
                       sharex=True, sharey=True)
color = [palette[0], palette[1], palette[2], palette[4]]

weight_list = np.arange(0, 1.1, 0.1, dtype=float)

witrs = []
wplist = []
agentplist = []


for s in range(4):
    df = this_df[this_df['scene']==s+1]

    sns.lineplot(data=df,x='weight',y='itrs_x', hue='agent',ax=ax[s], errorbar='se',\
                 palette=[color[s], color[s], color[s]], style='agent',\
                    dashes={agent_list[0]: custom_linestyles[0], 
                 agent_list[1]: custom_linestyles[1], 
                 agent_list[2]: custom_linestyles[2]}) # 
    
    agent_w_p = [ ] 
    agent_w_itrs = []
    for a in agent_list[1:]:
        w_p = []
        w_itr = []
        df_1 = df[df['agent']==a]
        for w in weight_list[1:-1]:
            this_comp_0 = df_1[(df_1['weight']==w)]['itrs_x']
            this_comp_1 = df_1[(df_1['weight']==0)]['itrs_x']
            this_comp_2 = df_1[(df_1['weight']==1)]['itrs_x']
            w_itr.append([this_comp_0, this_comp_1, this_comp_2])

            _, p1 = wilcoxon(x=this_comp_0, y=this_comp_1, alternative='greater')
            _, p2 = wilcoxon(x=this_comp_0, y=this_comp_2, alternative='greater')
            
            w_p.append(p1)
            w_p.append(p2)
        _, w_p, _, _ = multipletests(w_p, method='fdr_bh')
        w_p = w_p.reshape((len(weight_list[1:-1]),2))
        
        agent_w_itrs.append(w_itr)
        
        # print(w_p)
        for w in range(len(w_p)):
            if all(w_p[w]<0.05):
                ax[s].scatter(weight_list[w+1], np.mean(df_1[(df_1['weight']==weight_list[w+1])]['itrs_x']),\
                            marker='o', edgecolor=color[s], facecolors='white', s=15).set_zorder(3)
        agent_w_p.append(w_p)
    
    wplist.append(agent_w_p)


    if s == 0:
        ax[s].set_ylabel('Adaptation speed', fontsize=10)
        # ax[s].set_ylabel(r'Learning speed [${log_{10}(1/itr)}$]', fontsize=8)
        # '\\textit{Velocity (\N{DEGREE SIGN}/sec)}'
    else:
        ax[s].set_xlabel('')

    # Show tick bars at all values (e.g. 0.0 to 1.0 in your weight_list)
    ax[s].set_xticks(weight_list)  # tick bars

    # Show labels only at desired ticks
    label_locs = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    ax[s].set_xticklabels([str(round(w, 1)) if round(w, 1) in label_locs else '' for w in weight_list],\
                           fontsize=10)

    
    # ax[s].set_yticks(ax[s].get_yticks(),[round(y,1) for y in ax[s].get_yticks()], fontsize=10)
    ax[s].set_yticks(label_locs,label_locs, fontsize=10)
    # ax[s].set_ylim([-0.05,1])
    ax[s].legend().remove()

# handles, labels = ax[s].get_legend_handles_labels()
# plt.legend(handles,['Q learning', 'Weighted average', 'Synergy'],ncols=1, title='')
plt.tight_layout(w_pad=.1)
plt.gcf().set_dpi(1000)
plt.show()
fig.savefig('../../results/fig8a.svg', transparent=True, format='svg')

In [ ]:
# imperfect decoding_agent with CM
cm = 1/2.54
noise_ = 'CMsel'
prob_ = 1
weight_ = weight_list[1]

this_df = itr_df_2[(itr_df_2['noise']==noise_)\
                    &(itr_df_2['agent']!=agent_list[1])&(itr_df_2['transprob']==prob_)&\
                        (itr_df_2['weight']==weight_)]

fig, ax = plt.subplots(nrows=1, ncols=4, figsize=(4.5*4*cm,4.3*cm),\
                       sharex=True) 
color = [palette[0], palette[1], palette[2], palette[4]]


wplist = []
agentplist = []
witrs = []

for s in range(4):
    df = this_df[this_df['scene']==s+1]

    sns.pointplot(data=df,x='agent',y='itrs_x', ax=ax[s],\
                 color=color[s],errorbar='se') #color[s], color[s], color[s] 'black','black','black'
    
    this_comp_1 = df[(df['agent']==agent_list[0])]['itrs_x']
    this_comp_2 = df[(df['agent']==agent_list[-1])]['itrs_x']
    _, p = wilcoxon(x=this_comp_1, y=this_comp_2, alternative='two-sided')

    annots = [convert_pvalue_to_asterisks(p)]
    pairs = [(agent_list[0], agent_list[-1])]
    
    annot = Annotator(ax=ax[s], pairs=pairs, data=df, x='agent',y='itrs_x')
    annot.configure(loc='inside', fontsize=12,text_offset=0, line_width=.8, verbose=0)
    annot.set_custom_annotations(annots)
    annot.annotate()


    if s == 0:
        ax[s].set_ylabel('Adaptation speed', fontsize=10)
        # ax[s].set_ylabel(r'Learning speed [${log_{10}(1/itr)}$]', fontsize=8)
        # '\\textit{Velocity (\N{DEGREE SIGN}/sec)}'
    else:
        ax[s].set_ylabel('')

    ax[s].set_xlabel('')
    ax[s].set_xlim([-0.4, 1.4])

    ax[s].set_xticks([0,1], ['Beh.', 'NVA-syn.'],\
                           fontsize=10)

    if s == 0:
         ax[s].set_yticks([0,0.05,0.1], [0,0.05,0.1], fontsize=10)
         ax[s].set_ylim([0,0.12])
    elif s == 1:
         ax[s].set_yticks([0.01, 0.05, 0.09], [0.01, 0.05, 0.09], fontsize=10)
         ax[s].set_ylim([0.01, 0.09])
    elif s == 2:
         ax[s].set_yticks([0.05, 0.10, 0.15], [0.05, 0.10, 0.15], fontsize=10)
         ax[s].set_ylim([0.03, 0.15])
    elif s == 3:
         ax[s].set_yticks([0.10, 0.20], [0.10, 0.20], fontsize=10)
         ax[s].set_ylim([0.04, 0.22])


plt.tight_layout(w_pad=.1)
plt.gcf().set_dpi(1000)
plt.show()
fig.savefig('../../results/fig8b.svg', transparent=True, format='svg')

# Challenge 3

In [ ]:
# load results
with open(results_dir+'3.pkl', 'rb') as f:  # Python 3: open(..., 'wb')
    itr_df_3 = pickle.load(f)
with open(results_dir+'3_pes.pkl', 'rb') as f:  # Python 3: open(..., 'wb')
    itr_df_3_pes, gst_when = pickle.load(f)

In [ ]:
# decoding_agent with perfect decoding
cm = 1/2.54
prob_ = 1
stochGoal_ = 0.05
noise_ = 0

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(7*cm,6*cm),\
                       sharex=True, sharey=True) 
wplist = []
agentplist = []
witrs = []

palette0 = sns.color_palette("deep")
m = 3 # choose color

df = itr_df_3[(itr_df_3['noise']==noise_)\
            &(itr_df_3['transprob']==prob_)\
                &(itr_df_3['stochGoal']==stochGoal_)]

sns.lineplot(data=df,x='weight',y='ratio_x', hue='agent',ax=ax, errorbar='se',\
                palette=[palette0[m], palette0[m], palette0[m]], style='agent',\
                dashes={agent_list[0]: custom_linestyles[0], 
                agent_list[1]: custom_linestyles[1], 
                agent_list[2]: custom_linestyles[2]})
    

agent_w_p = [ ] 
for a in agent_list[2:]:
    print(f'Agent: {a}')
    w_p = []
    df_1 = df[df['agent']==a]
    for w in weight_list[1:-1]:
        this_comp_0 = df_1[(df_1['weight']==round(w,1))]['ratio_x']
        this_comp_1 = df_1[(df_1['weight']==0)]['ratio_x']
        this_comp_2 = df_1[(df_1['weight']==1)]['ratio_x']

        witrs.append([this_comp_0, this_comp_1, this_comp_2])
        _, p1 = wilcoxon(x=this_comp_0, y=this_comp_1, alternative='greater')
        _, p2 = wilcoxon(x=this_comp_0, y=this_comp_2, alternative='greater')
        
        w_p.append(p1)
        w_p.append(p2)


    _, w_p, _, _ = multipletests(w_p, method='fdr_bh')
    w_p = w_p.reshape((len(weight_list[1:-1]),2))
    wplist.append(w_p)
    for w in range(len(w_p)):
        if all(w_p[w]<0.05):
            ax.scatter(weight_list[w+1], np.mean(df_1[(df_1['weight']==round(weight_list[w+1],1))]['ratio_x']),\
                        marker='o', edgecolor=palette0[m], facecolors='white', s=15).set_zorder(3)
    agent_w_p.append(w_p)



    ax.set_ylabel('Adaptation speed', fontsize=12)
    # ax[s].set_ylabel(r'Learning speed [${log_{10}(1/itr)}$]', fontsize=8)
    # '\\textit{Velocity (\N{DEGREE SIGN}/sec)}'


    ax.set_xticks(weight_list,[round(w,1) for w in weight_list], fontsize=12)

    # Show labels only at desired ticks
    if noise_ == 'CM':
        ymin = 0.01
        ymax = 0.12
    elif noise_ == 0:
        ymin = 0.01
        ymax = 0.35
    
    ax.set_ylim([ymin, ymax])
    
    
    label_locs = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    ax.set_xticklabels([str(round(w, 1)) if round(w, 1) in label_locs else '' for w in weight_list],\
                            fontsize=12)
    
    ax.set_yticks([0,0.1,0.2,0.3],[0,0.1,0.2,0.3],\
                   fontsize=12)
    
    ax.legend().remove()


plt.tight_layout(w_pad=.1)
plt.gcf().set_dpi(1000)
plt.show()
fig.savefig('../../results/fig9a.svg', transparent=True, format='svg')


In [ ]:
# decoding_agent with certain noise_
cm = 1/2.54
prob_ = 1
stochGoal_ = 0.05
noise_ = 'CMsel'
w_ = weight_list[1]

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(7*cm,5.9*cm),\
                       sharex=True, sharey=True) #
wplist = []
agentplist = []
witrs = []

palette0 = sns.color_palette("deep")
m = 3 # choose color
df = itr_df_3[(itr_df_3['noise']==noise_)\
            &(itr_df_3['transprob']==prob_)&\
                (itr_df_3['stochGoal']==stochGoal_)&\
                    (itr_df_3['weight']==w_)&\
                        (itr_df_3['agent']!=agent_list[1])]

sns.pointplot(data=df,x='agent',y='ratio_x', ax=ax,\
                     color=palette0[m],errorbar='se') #color[s], color[s], color[s] 'black','black','black'
        
this_comp_1 = df[(df['agent']==agent_list[0])]['ratio_x']
this_comp_2 = df[(df['agent']==agent_list[-1])]['ratio_x']
_, p = wilcoxon(x=this_comp_1, y=this_comp_2, alternative='two-sided')
annots = [convert_pvalue_to_asterisks(p)]
pairs = [(agent_list[0], agent_list[-1])]

annot = Annotator(ax=ax, pairs=pairs, data=df, x='agent',y='ratio_x',\
                order=[agent_list[0], agent_list[-1]])
annot.configure(loc='inside', fontsize=12,text_offset=0, line_width=.8, verbose=0)
annot.set_custom_annotations(annots)
annot.annotate()

ax.set_ylabel('Adaptation speed', fontsize=12)
# ax.set_xlabel('')
ax.set_xlim([-0.4, 1.4])
ax.set_xticks([0,1], ['Beh.', 'NVA-syn.'],fontsize=12)
ax.set_yticks([0.01,0.02,0.03,0.04, 0.05], [0.01,0.02,0.03,0.04,0.05], fontsize=12)
ax.set_ylim([0.01,0.045])

# handles, labels = ax[s].get_legend_handles_labels()
# plt.legend(handles,['Q learning', 'Weighted average', 'Synergy'],ncols=1, title='')
plt.tight_layout(w_pad=.1)
plt.gcf().set_dpi(1000)
plt.show()
fig.savefig('../../results/fig9b.svg', transparent=True, format='svg')

In [ ]:
# Comparing different weights with perfect decoding_agent for 4 specific blocks with various stochGoal changes
this_weights = [0, 0.1, 1]

pes = ['rpe', 'spe']
fig, ax = plt.subplots(nrows=1, ncols=len(pes), \
                       figsize=(9*len(pes)*1/2.54, 5*1/2.54),\
                       )

# color = [palette[0], palette[1], palette[2], palette[4]]
color = [['black',RPE_COLOR_DARK, 'grey'],['black',SPE_COLOR_DARK, 'grey'], 'black']

wplist = []
agentplist = []
witrs = []

for pe in range(len(pes)):
    df = itr_df_3_pes
    
    sns.lineplot(data=df, x='trial',y=pes[pe], hue='weight', hue_order=this_weights,\
                 ax=ax[pe], errorbar='se',\
                 palette=color[pe], linewidth=1) #color[s], color[s], color[s] 'black','black','black'
    
    for t in range(len(gst_when)):
        if gst_when[t]:
            ax[pe].axvline(x=t, color='gray', linestyle='--', linewidth=0.7, alpha=0.5)

    if pe == 0:
        ax[pe].set_ylabel('RPEs', fontsize=12)
    elif pe == 1:
        ax[pe].set_ylabel('SPEs', fontsize=12)

    
    ax[pe].set_xticks(ax[pe].get_xticks(),[round(x) for x in ax[pe].get_xticks()], fontsize=12)
    
    # ax[pe,s].set_yticks(ax[pe,s].get_yticks(),[round(y,1) for y in ax[pe,s].get_yticks()], fontsize=10)
    ax[pe].set_xlim([0,100])
    ax[pe].set_ylim([-.1,ax[pe].get_ylim()[1]])
    ax[pe].legend().remove()


plt.tight_layout(w_pad=.1)
plt.gcf().set_dpi(1000)
plt.show()
fig.savefig('../../results/fig9c.svg', transparent=True, format='svg')